<a href="https://colab.research.google.com/github/Aje-dotcom/DeepTech-/blob/master/PdfQuestion_extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

the InclusiveBridge Certificate Simulator, you need a script that doesn't just "read" the PDF, but structurally parses it into a JSON format your React frontend can consume.
Because standard PDF libraries often struggle with the complex layouts of exam dumps (like columns and multiple-choice bubbles), I recommend using pdfplumber for its precision or PyMuPDF (fitz) for speed.
The "Hypersonic" Extraction Script
This script iterates through the PL-300 PDF, identifies the question patterns, and exports them into a structured simulator_db.json.

In [3]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.5 MB/s eta 0:00:00


In [7]:
import pdfplumber
import json
import re

def extract_exam_data(pdf_path, output_json):
    questions = []
    current_question = None

    # Regex patterns to identify Question start and Options
    q_pattern = re.compile(r'^(Question|Q)\s*#?\d+', re.IGNORECASE)
    opt_pattern = re.compile(r'^([A-E])\.\s')

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue

            lines = text.split('\n')
            for line in lines:
                line = line.strip()

                # Detect start of a new question
                if q_pattern.match(line):
                    if current_question:
                        questions.append(current_question)

                    current_question = {
                        "id": line,
                        "question": "",
                        "options": [],
                        "answer": ""
                    }

                # Detect options (A., B., etc.)
                elif opt_pattern.match(line) and current_question:
                    current_question["options"].append(line)

                # Detect the Correct Answer (usually marked as 'Answer:' or 'Correct Answer:')
                elif ("Answer:" in line or "Correct:" in line) and current_question:
                    current_question["answer"] = line.split(":")[-1].strip()

                # Append to the question body
                elif current_question and not opt_pattern.match(line):
                    # Avoid re-adding the ID line
                    if line not in current_question["id"]:
                        current_question["question"] += " " + line

        # Append the final question
        if current_question:
            questions.append(current_question)

    # Save to JSON for your Flask/React integration
    with open(output_json, 'w') as f:
        json.dump(questions, f, indent=4)
    print(f"Successfully extracted {len(questions)} questions to {output_json}")

# Execute for your PL-300 file
extract_exam_data("pl-300_merged.pdf", "pl300_data.json")

Successfully extracted 200 questions to pl300_data.json


In [1]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)

GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…